In [1]:


from scipy.signal import butter, filtfilt, find_peaks, welch
from scipy.integrate import simpson
from scipy.stats import skew, kurtosis
from scipy.stats import iqr

In [3]:
import os
import pickle
import numpy as np
from scipy.signal import resample
import pandas as pd

# --- FAST LOADING & CACHING CHECK ---
FEATURE_CSV_PATH = 'final_project_features.csv'
LOADED_PRECOMPUTED_FEATURES = False

if os.path.exists(FEATURE_CSV_PATH):
    print(f"[CACHE] Found pre-extracted features file: '{FEATURE_CSV_PATH}'")
    print("[CACHE] Loading features directly to skip raw data loading & 10+ min extraction loop...")
    features_df = pd.read_csv(FEATURE_CSV_PATH)
    LOADED_PRECOMPUTED_FEATURES = True
    print(f"[SUCCESS] Loaded pre-calculated feature matrix: {features_df.shape}")
else:
    print(f"[INFO] '{FEATURE_CSV_PATH}' not found locally. Will load raw WESAD signal files if available.")

def load_subject(subject_path):
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data

ecg_df = {}

if not LOADED_PRECOMPUTED_FEATURES:
    # Search for base WESAD path locally across OS platforms
    possible_base_paths = [
        '.', '..', 'C:/Users/tusha/Downloads/Miniproject/WESAD', '/home/tushar_verma/projects/WESAD'
    ]
    base_path = None
    for p in possible_base_paths:
        if os.path.exists(os.path.join(p, 'S2', 'S2.pkl')):
            base_path = p
            break

    if base_path:
        print(f"Loading raw WESAD signals from '{base_path}'...")
        for i in [2,3,5,6,7,8,9,10,11,13,14,15,16,17]:
            subject_file = os.path.join(base_path, f"S{i}", f"S{i}.pkl")
            if not os.path.exists(subject_file):
                continue
            data = load_subject(subject_file)
            ecg = data['signal']['chest']['ECG']
            ppg = data['signal']['wrist']['BVP']
            resp = data['signal']['chest']['Resp']
            labels = data['label']
            ppg_resampled = resample(ppg.flatten(), len(ecg))

            ecg_df[i] = pd.DataFrame({
                "ECG": ecg.flatten(),
                "PPG": ppg_resampled,
                "Resp": resp.flatten(),
                "labels": labels
            })
            del data
            print(f"Loaded S{i}: {ecg_df[i].shape}")
    else:
        print("[NOTICE] Raw WESAD subject pickle files not found in standard paths.")
else:
    print("Skipping raw signal loading.")


Loaded S2: (4255300, 4)
Loaded S3: (4545100, 4)
Loaded S5: (4380600, 4)
Loaded S6: (4949700, 4)
Loaded S7: (3666600, 4)
Loaded S8: (3826200, 4)
Loaded S9: (3656100, 4)
Loaded S10: (3847200, 4)
Loaded S11: (3663100, 4)
Loaded S13: (3875900, 4)
Loaded S14: (3883600, 4)
Loaded S15: (3676400, 4)
Loaded S16: (3941700, 4)
Loaded S17: (4144000, 4)


In [ ]:
# from ydata_profiling import ProfileReport



# RUN_PROFILING = False 
# SAMPLE_SIZE = 50_000    

# if RUN_PROFILING:
#     for i in [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]:
#         df_sample = ecg_df[i].sample(min(SAMPLE_SIZE, len(ecg_df[i])), random_state=42)
#         profile = ProfileReport(df_sample, title=f"Subject S{i} Profile", minimal=True)
#         profile.to_file(f"S{i}_profile.html")
#         print(f"Saved report for S{i}")
# else:
#     print("Skipping profiling (RUN_PROFILING=False) — proceed to the next cells.")

In [4]:
if not LOADED_PRECOMPUTED_FEATURES and ecg_df:
    for i in [2,3,5,6,7,8,9,10,11,13,14,15,16,17]:
        if i in ecg_df:
            df = ecg_df[i]
            ecg_df[i] = df[~df["labels"].isin([0,5,6, 7])]
            print(f"The shape of subject{i} is {ecg_df[i].shape}")
else:
    print("Skipping label filtering (precomputed features in use).")


The shape of  subject2 is (2022299, 4)
The shape of  subject3 is (2054501, 4)
The shape of  subject5 is (2107700, 4)
The shape of  subject6 is (2092300, 4)
The shape of  subject7 is (2091602, 4)
The shape of  subject8 is (2103499, 4)
The shape of  subject9 is (2093000, 4)
The shape of  subject10 is (2151100, 4)
The shape of  subject11 is (2113301, 4)
The shape of  subject13 is (2114700, 4)
The shape of  subject14 is (2114701, 4)
The shape of  subject15 is (2118899, 4)
The shape of  subject16 is (2109100, 4)
The shape of  subject17 is (2104900, 4)


In [5]:
if not LOADED_PRECOMPUTED_FEATURES and ecg_df:
    for i in [2,3,5,6,7,8,9,10,11,13,14,15,16,17]:
        if i not in ecg_df: continue
        df = ecg_df[i]
        print(f"S{i} — Original size:", df.shape)

        q1_ecg, q3_ecg = np.percentile(df["ECG"], [25, 75])
        iqr_ecg = q3_ecg - q1_ecg
        lower_ecg = q1_ecg - 1.5 * iqr_ecg
        upper_ecg = q3_ecg + 1.5 * iqr_ecg

        q1_ppg, q3_ppg = np.percentile(df["PPG"], [25, 75])
        iqr_ppg = q3_ppg - q1_ppg
        lower_ppg = q1_ppg - 1.5 * iqr_ppg
        upper_ppg = q3_ppg + 1.5 * iqr_ppg

        q1_resp, q3_resp = np.percentile(df["Resp"], [25, 75])
        iqr_resp = q3_resp - q1_resp
        lower_resp = q1_resp - 1.5 * iqr_resp
        upper_resp = q3_resp + 1.5 * iqr_resp

        ecg_df[i] = df[
            (df["ECG"] > lower_ecg) & (df["ECG"] < upper_ecg) &
            (df["PPG"] > lower_ppg) & (df["PPG"] < upper_ppg) &
            (df["Resp"] > lower_resp) & (df["Resp"] < upper_resp)
        ].reset_index(drop=True)

        print(f"S{i} — After removing outliers:", ecg_df[i].shape)
else:
    print("Skipping outlier removal (precomputed features in use).")


S2 — Original size: (2022299, 4)
S2 — After removing outliers: (1651529, 4)
S3 — Original size: (2054501, 4)
S3 — After removing outliers: (1596432, 4)
S5 — Original size: (2107700, 4)
S5 — After removing outliers: (1446245, 4)
S6 — Original size: (2092300, 4)
S6 — After removing outliers: (1584015, 4)
S7 — Original size: (2091602, 4)
S7 — After removing outliers: (1714274, 4)
S8 — Original size: (2103499, 4)
S8 — After removing outliers: (1638108, 4)
S9 — Original size: (2093000, 4)
S9 — After removing outliers: (1791192, 4)
S10 — Original size: (2151100, 4)
S10 — After removing outliers: (1712293, 4)
S11 — Original size: (2113301, 4)
S11 — After removing outliers: (1508812, 4)
S13 — Original size: (2114700, 4)
S13 — After removing outliers: (1696770, 4)
S14 — Original size: (2114701, 4)
S14 — After removing outliers: (1760145, 4)
S15 — Original size: (2118899, 4)
S15 — After removing outliers: (1564948, 4)
S16 — Original size: (2109100, 4)
S16 — After removing outliers: (1669161, 4)


In [6]:
Fs=700
win_sec=60
step_sec=30
purity_threshold=0.9

window_size=win_sec=win_sec*Fs
step_size=step_sec*Fs

window=[]

if not LOADED_PRECOMPUTED_FEATURES and ecg_df:
    for subject_id,df in ecg_df.items():
        n=len(df)
        for start in range(0,n-window_size+1,step_size):
            chunk=df.iloc[start:start+window_size]
            label_count=chunk["labels"].value_counts(normalize=True)
            majority_lab=label_count.idxmax()
            purity=label_count.max()
            if(purity<purity_threshold):
                continue
            window.append({
                "subject": subject_id,
                "ecg": chunk["ECG"].values,
                "ppg": chunk["PPG"].values,
                "resp": chunk["Resp"].values,
                "labels": majority_lab}
            )
    print(f"Generated {len(window)} signal windows.")
else:
    print("Skipping window segmentation (precomputed features in use).")


982


In [7]:
hr = np.array([np.nan])
rr = np.array([])
rr_ms = np.array([])
pulse_rate = np.array([np.nan])
pulse_interval = np.array([])
lf_power = hf_power = vlf_power = total = 0.0
amo = mo = mxdmn = np.nan

In [8]:
try:
    import antropy as ant
    from scipy.stats import skew, kurtosis, iqr
    import pywt
    import pycatch22
    from scipy.signal import find_peaks, welch, butter, filtfilt
    from scipy.integrate import simpson
    import numpy as np
except ImportError as e:
    if 'LOADED_PRECOMPUTED_FEATURES' in locals() and LOADED_PRECOMPUTED_FEATURES:
        print(f"[NOTICE] Feature extraction package ({e}) not installed, but precomputed features are loaded. Notebook will proceed using cached features.")
    else:
        print(f"[WARNING] Missing package for raw extraction: {e}")

def extract_catch22_features(signal_array, prefix):
    if 'pycatch22' not in globals(): return {}
    result = pycatch22.catch22_all(signal_array)
    return {f"{prefix}_catch22_{name}": val for name, val in zip(result['names'], result['values'])}

def extract_resp_features(resp, fs=700):
    resp = np.array(resp, dtype=np.float64, copy=True)
    features = {}
    b, a = butter(2, [0.1, 0.5], btype="band", fs=fs)
    resp_filt = filtfilt(b, a, resp)
    min_distance = int(fs * 1.5)
    peaks, _ = find_peaks(resp_filt, distance=min_distance)
    if len(peaks) > 2:
        breath_intervals = np.diff(peaks) / fs
        breathing_rate = 60 / breath_intervals
        features["resp_rate_mean"] = np.mean(breathing_rate)
        features["resp_rate_std"] = np.std(breathing_rate)
        features["resp_interval_cv"] = np.std(breath_intervals) / (np.mean(breath_intervals) + 1e-8)
        features["resp_interval_iqr"] = iqr(breath_intervals)
    else:
        features["resp_rate_mean"] = np.nan
        features["resp_rate_std"] = np.nan
        features["resp_interval_cv"] = np.nan
        features["resp_interval_iqr"] = np.nan
    features["resp_amplitude_std"] = np.std(resp_filt)
    features["resp_amplitude_skew"] = skew(resp_filt)
    features["resp_amplitude_kurtosis"] = kurtosis(resp_filt)
    return features


In [9]:
from joblib import Parallel, delayed
import traceback

def process_window(w):
    try:
        feats = extract_features(w["ecg"], w["ppg"], w.get("resp"))
        feats["subject"] = w["subject"]
        feats["label"] = w["labels"]
        return feats
    except Exception as e:
        print(f"Feature extraction failed for Subject {w['subject']}: {e}")
        traceback.print_exc()
        return None


In [10]:
if not LOADED_PRECOMPUTED_FEATURES and window:
    test_w = window[0]
    print("Testing window for subject:", test_w["subject"])
    try:
        feats = extract_features(test_w["ecg"], test_w["ppg"], test_w["resp"])
        print("SUCCESS. Number of features:", len(feats))
    except Exception:
        import traceback
        traceback.print_exc()
else:
    print("Skipping single window debug test.")


Testing window for subject: 2
ecg len: 42000 dtype: float64
ppg len: 42000 dtype: float64
resp len: 42000 dtype: float64
SUCCESS. Number of features: 86
{'heart_rate': np.float64(108.79732406896248), 'mean_hr': np.float64(108.79732406896248), 'mean_rr': np.float64(612.8129602356406), 'median_rr': np.float64(634.2857142857142), 'sdnn': np.float64(178.42078518985818), 'rmssd': np.float64(179.68983102599228), 'sdsd': np.float64(179.67941727359883), 'nn20': np.int64(81), 'nn50': np.int64(69), 'pnn20': np.float64(0.84375), 'pnn50': np.float64(0.71875), 'cvnn': np.float64(0.29115047619301543), 'rr_variance': np.float64(31833.976587765515), 'rr_range': np.float64(790.0), 'lf_power': 0.0047856123356306785, 'hf_power': 0.00864381605793071, 'vlf_power': 0.0, 'total_power': 0.013429428393561389, 'lf_hf_ratio': 0.5536457796178225, 'lf_nu': 0.35635264284888185, 'hf_nu': 0.6436473497047849, 'peak_frequency': np.float64(0.0625), 'spectral_entropy': np.float64(4.2175589596198995), 'psd_mean': np.float

In [11]:
if not LOADED_PRECOMPUTED_FEATURES:
    from scipy.signal import butter, filtfilt, find_peaks, welch
    from scipy.integrate import simpson
    feature_rows = Parallel(
        n_jobs=-1,
        verbose=10
    )(
        delayed(process_window)(w)
        for w in window
    )
    feature_rows = [r for r in feature_rows if r is not None]
    features_df = pd.DataFrame(feature_rows)
    print("Feature Matrix Shape:", features_df.shape)
else:
    print(f"[CACHE] Using pre-loaded feature matrix: {features_df.shape}")


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   19.1s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   20.0s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:   29.1s
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   31.5s
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:   39.6s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:   46.0s
[Parallel(n_jobs=-1)]: Done  61 tasks      | elapsed:   55.8s
[Parallel(n_jobs=-1)]: Done  74 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done  89 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 104 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 121 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 197 tasks      | elapsed:  

Feature Matrix Shape: (982, 88)


[Parallel(n_jobs=-1)]: Done 982 out of 982 | elapsed: 10.1min finished


In [12]:
if 'features_df' in locals() and not LOADED_PRECOMPUTED_FEATURES:
    constant_cols = [
        c for c in features_df.columns
        if c not in ["subject", "label"]
        and features_df[c].nunique() <= 1
    ]
    print(f"Removing {len(constant_cols)} constant features")
    features_df.drop(columns=constant_cols, inplace=True)
else:
    print(f"Loaded clean features shape: {features_df.shape}")


Removing 1 constant features


In [13]:
features_df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [14]:
na_counts = features_df.isna().sum()

na_counts = na_counts[na_counts > 0]

print("Columns containing NaNs")

print(na_counts.sort_values(ascending=False))

Columns containing NaNs
Series([], dtype: int64)


In [15]:
missing_ratio = features_df.isna().mean()

bad_columns = missing_ratio[
    missing_ratio > 0.30
].index

print("Removing columns:")

print(list(bad_columns))

features_df.drop(columns=bad_columns, inplace=True)

Removing columns:
[]


In [16]:
feature_cols = [
    c for c in features_df.columns
    if c not in ["subject", "label"]
]

features_df[feature_cols] = (
    features_df[feature_cols]
    .fillna(features_df[feature_cols].median())
)

In [17]:
print("Final Shape:", features_df.shape)

print(
    "Remaining NaNs:",
    features_df.isna().sum().sum()
)

print(features_df.head())

Final Shape: (982, 87)
Remaining NaNs: 0
   heart_rate     mean_hr     mean_rr   median_rr        sdnn       rmssd  \
0  108.797324  108.797324  612.812960  634.285714  178.420785  179.689831   
1   98.994409   98.994409  658.351648  708.571429  157.906511  162.688161   
2   99.759238   99.759238  649.006211  693.571429  152.702618  176.158008   
3   97.224293   97.224293  668.198052  702.857143  164.529956  192.015618   
4   93.245820   93.245820  690.215947  733.571429  142.458000  155.691155   

         sdsd  nn20  nn50     pnn20  ...   ppg_sqi  resp_rate_mean  \
0  179.679417    81    69  0.843750  ...  1.533333       23.639294   
1  162.688049    80    62  0.888889  ...  1.466667       23.944914   
2  176.157336    78    58  0.857143  ...  1.450000       25.972361   
3  192.012919    74    54  0.850575  ...  1.350000       25.963075   
4  155.680377    74    43  0.870588  ...  1.300000       24.923074   

   resp_rate_std  resp_interval_cv  resp_interval_iqr  resp_amplitude_std  

In [18]:
features_df.to_csv('final_project_features.csv', index=False)
print('Feature CSV saved to final_project_features.csv')


Feature CSV saved.


In [19]:

non_feature_cols = ["subject", "label"]
feature_cols = [c for c in features_df.columns if c not in non_feature_cols]

print(f"Using {len(feature_cols)} feature columns")

normalized_rows = []

for subject_id, group in features_df.groupby("subject"):

    
    baseline = group[group["label"] == 1]

    g = group.copy()

    if len(baseline) > 0:

        baseline_mean = baseline[feature_cols].mean()
        baseline_std = baseline[feature_cols].std()

        
        g[feature_cols] = (
            g[feature_cols] - baseline_mean
        ) / (baseline_std + 1e-8)

    normalized_rows.append(g)

features_df_norm = pd.concat(normalized_rows, ignore_index=True)

print(features_df_norm["label"].value_counts())

Using 85 feature columns
label
1    411
4    267
2    180
3    124
Name: count, dtype: int64


In [20]:
print(features_df.columns.tolist())

['heart_rate', 'mean_hr', 'mean_rr', 'median_rr', 'sdnn', 'rmssd', 'sdsd', 'nn20', 'nn50', 'pnn20', 'pnn50', 'cvnn', 'rr_variance', 'rr_range', 'lf_power', 'hf_power', 'total_power', 'lf_hf_ratio', 'lf_nu', 'hf_nu', 'peak_frequency', 'spectral_entropy', 'psd_mean', 'psd_std', 'ecg_wav_energy_L0', 'ecg_wav_energy_L1', 'ecg_wav_energy_L2', 'ecg_wav_energy_L3', 'ecg_wav_energy_L4', 'ppg_wav_energy_L0', 'ppg_wav_energy_L1', 'ppg_wav_energy_L2', 'ppg_wav_energy_L3', 'ppg_wav_energy_L4', 'ecg_app_entropy', 'ecg_svd_entropy', 'ecg_spectral_entropy', 'ecg_higuchi_fd', 'ecg_katz_fd', 'ecg_detrended_fluctuation', 'ecg_sampen', 'ecg_perm_entropy', 'ecg_hjorth_mobility', 'ecg_hjorth_complexity', 'ppg_sampen', 'ppg_perm_entropy', 'ppg_hjorth_mobility', 'ppg_hjorth_complexity', 'ecg_mean', 'ecg_std', 'ppg_mean', 'ppg_std', 'ppg_min', 'ppg_max', 'pulse_rate', 'pulse_interval_mean', 'pulse_interval_std', 'pulse_amplitude', 'pulse_amplitude_std', 'ppg_variance', 'ppg_rms', 'ppg_skewness', 'ppg_kurtosis

In [21]:
print(features_df.head())

   heart_rate     mean_hr     mean_rr   median_rr        sdnn       rmssd  \
0  108.797324  108.797324  612.812960  634.285714  178.420785  179.689831   
1   98.994409   98.994409  658.351648  708.571429  157.906511  162.688161   
2   99.759238   99.759238  649.006211  693.571429  152.702618  176.158008   
3   97.224293   97.224293  668.198052  702.857143  164.529956  192.015618   
4   93.245820   93.245820  690.215947  733.571429  142.458000  155.691155   

         sdsd  nn20  nn50     pnn20  ...   ppg_sqi  resp_rate_mean  \
0  179.679417    81    69  0.843750  ...  1.533333       23.639294   
1  162.688049    80    62  0.888889  ...  1.466667       23.944914   
2  176.157336    78    58  0.857143  ...  1.450000       25.972361   
3  192.012919    74    54  0.850575  ...  1.350000       25.963075   
4  155.680377    74    43  0.870588  ...  1.300000       24.923074   

   resp_rate_std  resp_interval_cv  resp_interval_iqr  resp_amplitude_std  \
0       5.588634          0.232027     

In [22]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

# use the normalized feature set (features_df_norm) — swap to features_df if you didn't run the normalization cell
data_for_model = features_df_norm

non_feature_cols = ["subject", "label"]
feature_cols = [c for c in data_for_model.columns if c not in non_feature_cols]

x = data_for_model[feature_cols]
groups = data_for_model["subject"]

le = LabelEncoder()
y = le.fit_transform(data_for_model["label"])

gkf = GroupKFold(n_splits=5)

# placeholder params — replace with your tuned values if you have them
best_params = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8
}

print("x shape:", x.shape)
print("classes:", le.classes_)


x shape: (982, 85)
classes: [1 2 3 4]


In [23]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import pandas as pd
import joblib

gkf = GroupKFold(n_splits=5)

fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
all_y_true = []
all_y_pred = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(x, y, groups=groups)):
    x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

    model = XGBClassifier(**best_params, objective="multi:softmax", num_class=len(np.unique(y)), random_state=42, eval_metric="mlogloss", tree_method="hist")
    model.fit(x_train, y_train, sample_weight=sample_weights)
    y_pred = model.predict(x_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    fold_accuracies.append(acc)
    fold_precisions.append(prec)
    fold_recalls.append(rec)
    fold_f1s.append(f1)

    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

    print(f"Fold {fold + 1} — Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

print(f"Accuracy:  {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Precision: {np.mean(fold_precisions):.4f} ± {np.std(fold_precisions):.4f}")
print(f"Recall:    {np.mean(fold_recalls):.4f} ± {np.std(fold_recalls):.4f}")
print(f"F1:        {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

print(classification_report(all_y_true, all_y_pred, target_names=le.classes_.astype(str)))

cm = confusion_matrix(all_y_true, all_y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_.astype(str), columns=le.classes_.astype(str))
print(cm_df)

# Save trained final model artifact so it does not need to be retrained on another laptop
print("Training final XGBoost model on all samples for saving...")
final_model = XGBClassifier(**best_params, objective="multi:softmax", num_class=len(np.unique(y)), random_state=42, eval_metric="mlogloss", tree_method="hist")
sample_weights_all = compute_sample_weight(class_weight="balanced", y=y)
final_model.fit(x, y, sample_weight=sample_weights_all)
joblib.dump(final_model, "final_project_model.joblib", compress=3)
print("[SUCCESS] Trained model saved to 'wesad_model.joblib'.")


Fold 1 — Accuracy: 0.7282 | Precision: 0.8209 | Recall: 0.5655 | F1: 0.5864
Fold 2 — Accuracy: 0.8707 | Precision: 0.8579 | Recall: 0.8285 | F1: 0.8400
Fold 3 — Accuracy: 0.8294 | Precision: 0.7953 | Recall: 0.7335 | F1: 0.7562
Fold 4 — Accuracy: 0.8558 | Precision: 0.8736 | Recall: 0.7937 | F1: 0.8133
Fold 5 — Accuracy: 0.8238 | Precision: 0.8382 | Recall: 0.7434 | F1: 0.7670
Accuracy:  0.8216 ± 0.0498
Precision: 0.8372 ± 0.0275
Recall:    0.7329 ± 0.0905
F1:        0.7526 ± 0.0885
              precision    recall  f1-score   support

           1       0.79      0.97      0.87       411
           2       0.88      0.63      0.74       180
           3       0.76      0.42      0.54       124
           4       0.87      0.90      0.88       267

    accuracy                           0.82       982
   macro avg       0.82      0.73      0.76       982
weighted avg       0.82      0.82      0.81       982

     1    2   3    4
1  398    5   3    5
2   41  114  11   14
3   48    6  5

In [24]:
# NOTE: sample_weight balancing is now applied inside the fold loop above.
# This cell is kept for reference but is no longer needed to run separately.
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
model.fit(x_train, y_train, sample_weight=sample_weights)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import loa

In [25]:
subject_results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(x, y, groups=groups)):
    x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    test_subjects = groups.iloc[test_idx]

    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

    model = XGBClassifier(**best_params, objective="multi:softmax", num_class=len(np.unique(y)), random_state=42, eval_metric="mlogloss", tree_method="hist")
    model.fit(x_train, y_train, sample_weight=sample_weights)
    y_pred = model.predict(x_test)

    results_df = pd.DataFrame({
        "subject": test_subjects.values,
        "y_true": y_test,
        "y_pred": y_pred,
        "fold": fold + 1
    })
    subject_results.append(results_df)

all_results = pd.concat(subject_results, ignore_index=True)

per_subject_acc = all_results.groupby("subject").apply(
    lambda g: accuracy_score(g["y_true"], g["y_pred"])
).sort_values()

print("Per-Subject Accuracy (lowest first):")
print(per_subject_acc)

per_subject_class3_recall = all_results[all_results["y_true"] == le.transform([3])[0] if 3 in le.classes_ else None]

Per-Subject Accuracy (lowest first):
subject
9     0.620253
10    0.657534
3     0.716418
11    0.746032
15    0.746269
2     0.800000
8     0.800000
13    0.833333
5     0.850000
6     0.910448
17    0.914286
14    0.935065
7     0.959459
16    0.972603
dtype: float64


/tmp/ipykernel_727/1882843329.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_subject_acc = all_results.groupby("subject").apply(


In [26]:
low_acc_subjects = per_subject_acc[per_subject_acc < 0.70].index.tolist()

subject_class_breakdown = all_results[all_results["subject"].isin(low_acc_subjects)]

for subj in low_acc_subjects:
    subj_data = subject_class_breakdown[subject_class_breakdown["subject"] == subj]
    print(f"\nSubject {subj}:")
    print(classification_report(subj_data["y_true"], subj_data["y_pred"], zero_division=0))


Subject 9:
              precision    recall  f1-score   support

           0       0.52      1.00      0.68        31
           1       1.00      0.12      0.22        16
           2       0.50      0.10      0.17        10
           3       1.00      0.68      0.81        22

    accuracy                           0.62        79
   macro avg       0.75      0.48      0.47        79
weighted avg       0.75      0.62      0.56        79


Subject 10:
              precision    recall  f1-score   support

           0       0.70      1.00      0.82        28
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00         9
           3       0.61      1.00      0.75        20

    accuracy                           0.66        73
   macro avg       0.33      0.50      0.39        73
weighted avg       0.43      0.66      0.52        73



In [27]:
# Check class 2's overall representation and how consistent its features are across subjects
class2_data = x[y == 2]
class2_subjects = groups[y == 2]

class2_data.groupby(class2_subjects).mean().T  # per-subject means for class-2 samples

subject,2,3,5,6,7,8,9,10,11,13,14,15,16,17
heart_rate,-2.669753,-1.366664,1.292601,-0.385051,-2.120478,0.465467,-0.399022,0.089097,0.870783,-1.151775,-2.259870,-1.487646,-3.080445,-2.749606
mean_hr,-2.669753,-1.366664,1.292601,-0.385051,-2.120478,0.465467,-0.399022,0.089097,0.870783,-1.151775,-2.259870,-1.487646,-3.080445,-2.749606
mean_rr,3.384563,1.135479,-0.777645,0.673172,2.275183,-0.532828,-0.406690,1.601382,-0.998888,1.210963,2.198280,1.947866,1.555771,2.669763
median_rr,3.247165,0.636682,-0.797008,1.000311,3.159778,-0.735921,-0.943769,1.829713,-0.951839,1.051354,2.093322,2.077348,0.946161,2.665727
sdnn,-0.290888,0.628172,0.887357,0.541691,1.472138,0.296474,-3.372148,4.497368,-0.472315,-0.418388,0.934112,0.366685,-6.598529,0.810657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
resp_interval_cv,-0.238049,0.624813,0.492775,0.078292,0.334559,0.522062,-1.077730,-0.823958,2.803553,-0.434639,-0.083699,-0.489108,0.012607,0.294440
resp_interval_iqr,0.403806,-0.080552,0.256528,-0.358872,0.017570,0.654142,-1.127592,-0.323248,5.097055,-1.059355,-0.267162,0.256050,-0.677585,0.606054
resp_amplitude_std,4.545343,1.008016,-2.178995,-0.018318,0.371038,3.293394,2.323955,0.929393,2.349222,0.146443,2.115832,-3.430514,0.254093,1.831956
resp_amplitude_skew,1.298794,-0.680053,-1.455878,-1.419636,-0.523590,-0.681316,-0.161246,0.006195,1.549995,0.382372,0.678023,-1.426298,-0.185894,-0.306941
